In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv("../data/raw/customer_churn.csv")
df.head()

,CustomerID,Age,Gender,Annual_Income_USD,Spending_Score,Membership_Status,Preferred_Payment_Method,Region,Total_Purchases,Avg_Purchase_Value,Last_Purchase_Date,Churn,Satisfaction_Score,Website_Visits_Last_Month,Avg_Time_Per_Visit_Minutes,Support_Tickets_Last_6_Months,Referred_Friends
0,2f8c8c58-4779-4006-aa23-81db3a352da3,56,Female,79228,73,Bronze,Cryptocurrency,Central,17,209.07,2025-05-01,0,2.3,7,3.01,0,1
1,4e6558a6-7f45-4f2a-8dfc-fb05210c5a54,69,Female,23205,65,Silver,Cryptocurrency,Central,21,25.60,2025-04-14,0,2.5,14,5.80,5,2
2,11ab8d14-6dd9-4a49-84cc-963431a8e6fc,46,Male,54929,68,Bronze,Debit Card,Central,25,105.48,2024-11-20,0,4.6,18,5.00,0,0
3,503e4ec0-42c7-4c2d-8cda-a480ad492b4b,32,Female,103384,71,Bronze,PayPal,Central,25,381.95,2024-08-22,1,4.9,20,5.98,5,0
4,841426ba-7371-43b1-9cba-036d07d6c85e,60,Male,53411,11,Silver,PayPal,West,24,319.19,2025-02-05,0,1.2,18,1.67,1,1


In [3]:
#Convert date + create Days_Since_Last_Purchase
df['Last_Purchase_Date'] = pd.to_datetime(df['Last_Purchase_Date'])

latest_date = df['Last_Purchase_Date'].max()

df['Days_Since_Last_Purchase'] = (latest_date - df['Last_Purchase_Date']).dt.days


In [4]:
df_fe = df.copy()
df_fe[['Last_Purchase_Date', 'Days_Since_Last_Purchase']].head()


,Last_Purchase_Date,Days_Since_Last_Purchase
0,2025-05-01,50
1,2025-04-14,67
2,2024-11-20,212
3,2024-08-22,302
4,2025-02-05,135


In [5]:
df_fe.shape, df_fe['Days_Since_Last_Purchase'].min(), df_fe['Days_Since_Last_Purchase'].max()


((9000, 18), 0, 365)

In [6]:
# Engagement score
df_fe['Engagement_Score'] = (
    df_fe['Website_Visits_Last_Month'] * 0.4 +
    df_fe['Avg_Time_Per_Visit_Minutes'] * 0.4 +
    df_fe['Referred_Friends'] * 0.2
)


In [7]:
df_fe['Engagement_Score'].describe()


count    9000.000000
mean        8.168401
std         1.741364
min         1.868000
25%         6.976000
50%         8.136000
75%         9.292000
max        15.680000
Name: Engagement_Score, dtype: float64

In [8]:
# Customer Value Score
df_fe['Customer_Value_Score'] = (
    df_fe['Total_Purchases'] * 0.5 +
    df_fe['Avg_Purchase_Value'] * 0.5
)


In [9]:
df_fe['Customer_Value_Score'].describe()


count    9000.000000
mean      137.030913
std        71.232913
min        10.405000
25%        75.520000
50%       136.317500
75%       199.010000
max       264.230000
Name: Customer_Value_Score, dtype: float64

In [10]:
# Service Risk Indicator
df_fe['Service_Risk_Score'] = (
    df_fe['Support_Tickets_Last_6_Months'] * 0.7 +
    (5 - df_fe['Satisfaction_Score']) * 0.3
)


In [11]:
df_fe['Service_Risk_Score'].describe()

count    9000.000000
mean        1.643634
std         0.912327
min         0.000000
25%         0.970000
50%         1.570000
75%         2.250000
max         6.440000
Name: Service_Risk_Score, dtype: float64

### Engineered Feature Validation
- Engagement_Score shows a well-distributed range, capturing varying levels of customer interaction.
- Customer_Value_Score exhibits wide variance, enabling identification of high-value customers.
- Service_Risk_Score highlights a small but critical segment of customers with elevated churn risk.
- These features provide higher-level behavioral signals suitable for predictive modeling.


In [12]:
categorical_cols = [
    'Gender',
    'Membership_Status',
    'Preferred_Payment_Method',
    'Region'
]

categorical_cols

['Gender', 'Membership_Status', 'Preferred_Payment_Method', 'Region']

In [13]:
df_encoded = pd.get_dummies(
    df_fe,
    columns=categorical_cols,
    drop_first=True
)

In [14]:
df_encoded.shape

(9000, 29)

In [15]:
df_encoded.columns

Index(['CustomerID', 'Age', 'Annual_Income_USD', 'Spending_Score',
       'Total_Purchases', 'Avg_Purchase_Value', 'Last_Purchase_Date', 'Churn',
       'Satisfaction_Score', 'Website_Visits_Last_Month',
       'Avg_Time_Per_Visit_Minutes', 'Support_Tickets_Last_6_Months',
       'Referred_Friends', 'Days_Since_Last_Purchase', 'Engagement_Score',
       'Customer_Value_Score', 'Service_Risk_Score', 'Gender_Male',
       'Gender_Other', 'Membership_Status_Gold', 'Membership_Status_Platinum',
       'Membership_Status_Silver', 'Preferred_Payment_Method_Cryptocurrency',
       'Preferred_Payment_Method_Debit Card',
       'Preferred_Payment_Method_PayPal', 'Region_East', 'Region_North',
       'Region_South', 'Region_West'],
      dtype='object')

In [17]:
df_encoded.head(2)

,CustomerID,Age,Annual_Income_USD,Spending_Score,Total_Purchases,Avg_Purchase_Value,Last_Purchase_Date,Churn,Satisfaction_Score,Website_Visits_Last_Month,...,Membership_Status_Gold,Membership_Status_Platinum,Membership_Status_Silver,Preferred_Payment_Method_Cryptocurrency,Preferred_Payment_Method_Debit Card,Preferred_Payment_Method_PayPal,Region_East,Region_North,Region_South,Region_West
0,2f8c8c58-4779-4006-aa23-81db3a352da3,56,79228,73,17,209.07,2025-05-01,0,2.3,7,...,False,False,False,True,False,False,False,False,False,False
1,4e6558a6-7f45-4f2a-8dfc-fb05210c5a54,69,23205,65,21,25.60,2025-04-14,0,2.5,14,...,False,False,True,True,False,False,False,False,False,False


In [18]:
df_model = df_encoded.drop(
    columns=['CustomerID', 'Last_Purchase_Date']
)


In [19]:
X = df_model.drop(columns=['Churn'])
y = df_model['Churn']

In [24]:
X.shape, y.shape

((9000, 26), (9000,))

In [21]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

In [22]:
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

In [23]:
X.describe().T.head()

,count,mean,std,min,25%,50%,75%,max
Age,9000.0,2.368476e-18,1.000056,-1.713825,-0.841835,-0.036922,0.835068,1.707057
Annual_Income_USD,9000.0,-9.947598e-17,1.000056,-1.724380,-0.889739,0.012144,0.861275,1.728269
Spending_Score,9000.0,-3.157968e-17,1.000056,-1.699573,-0.863104,-0.026635,0.879539,1.716008
Total_Purchases,9000.0,1.744777e-16,1.000056,-3.413210,-0.684986,-0.002930,0.679126,4.544110
Avg_Purchase_Value,9000.0,-1.776357e-17,1.000056,-1.713464,-0.861082,-0.014741,0.872773,1.726963


### Feature Engineering & Data Preparation Summary

In this phase, the dataset was transformed into a machine-learning-ready format through systematic feature engineering and preprocessing steps.

Key engineered features were created to capture higher-level customer behavior:
- **Engagement_Score** combined website visits, session duration, and referral activity to represent customer interaction intensity.
- **Customer_Value_Score** summarized customer revenue contribution using purchase frequency and average transaction value.
- **Service_Risk_Score** quantified churn risk arising from service friction by combining support ticket volume with inverted satisfaction ratings.

Categorical variables (gender, membership status, payment method, and region) were converted into numerical form using **one-hot encoding**, ensuring no artificial ordinal relationships were introduced. Redundant categories were dropped to prevent multicollinearity.

Numerical features were standardized using **z-score normalization (StandardScaler)** to place all variables on a comparable scale, improving model stability and convergence for distance- and gradient-based algorithms.

After preprocessing:
- The final feature matrix consisted of **9,000 observations and 26 features**
- All features were numeric, well-scaled, and suitable for predictive modeling

This structured feature engineering process significantly improved signal quality and prepared the dataset for effective churn prediction modeling.
